# 🧠 Neuronales Netz von Grund auf — nur mit NumPy

**Ziel dieses Notebooks:** Wir bauen ein vollständiges neuronales Netzwerk **von Grund auf** — ohne TensorFlow, ohne PyTorch. Nur mit **NumPy** und Mathematik.

## Was wir hier lernen:
1. **Architektur** — Dense Layer, Aktivierungsfunktionen (ReLU, Sigmoid, Softmax)
2. **Forward-Pass** — Wie fließen die Daten durch das Netz?
3. **Backpropagation** — Wie werden die Gradienten berechnet?
4. **Training** — SGD mit Momentum auf dem MNIST-Datensatz

Alle Komponenten stammen aus `nn_core.py` — unserer selbstgebauten Neural-Network-Bibliothek.

In [ ]:
import sys
import os

# Stelle sicher, dass wir nn_core importieren können
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".")

import numpy as np
import matplotlib.pyplot as plt

# Unsere selbstgebaute Bibliothek
from nn_core import (
    Dense, ReLU, Sigmoid, Softmax,
    CrossEntropyLoss, SGD, NeuralNetwork
)

print("✅ Alle Module geladen!")
print(f"   NumPy Version: {np.__version__}")

---
## 1. Aktivierungsfunktionen

Aktivierungsfunktionen bringen **Nichtlinearität** ins Netzwerk. Ohne sie wäre ein mehrschichtiges Netz nur eine lineare Transformation.

### ReLU — Rectified Linear Unit
$$f(x) = \max(0, x)$$

Die am häufigsten verwendete Aktivierungsfunktion. Einfach, schnell und effektiv.

### Sigmoid
$$f(x) = \frac{1}{1 + e^{-x}}$$

Quetscht Werte in den Bereich (0, 1). Früher Standard, heute meist durch ReLU ersetzt.

### Softmax
$$f(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Wandelt Logits in Wahrscheinlichkeiten um (Ausgabeschicht für Klassifikation).

In [ ]:
# ── Aktivierungsfunktionen live ausprobieren ──
x = np.linspace(-5, 5, 100).reshape(-1, 1)

relu = ReLU()
sigmoid = Sigmoid()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ReLU
axes[0].plot(x, relu.forward(x), 'b-', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('ReLU: f(x) = max(0, x)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)

# Sigmoid
axes[1].plot(x, sigmoid.forward(x), 'g-', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Sigmoid: f(x) = 1/(1+e⁻ˣ)')
axes[1].set_xlabel('x')
axes[1].grid(True, alpha=0.3)

# Softmax (2-Klassen-Beispiel)
x2 = np.column_stack([x, -x])  # Zwei Klassen
sm = Softmax()
probs = sm.forward(x2)
axes[2].plot(x, probs[:, 0], 'r-', linewidth=2, label='Klasse 0')
axes[2].plot(x, probs[:, 1], 'orange', linewidth=2, label='Klasse 1')
axes[2].set_title('Softmax (2 Klassen)')
axes[2].set_xlabel('x')
axes[2].set_ylabel('P(Klasse)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Gradienten der Aktivierungsfunktionen

Für die Backpropagation brauchen wir die Ableitungen:

- **ReLU:** $f'(x) = 1$ wenn $x > 0$, sonst $0$
- **Sigmoid:** $f'(x) = f(x) \cdot (1 - f(x))$
- **Softmax+CrossEntropy:** Kombinierter Gradient (numerisch stabiler)

In [ ]:
# ── Gradienten visualisieren ──
x = np.linspace(-5, 5, 100).reshape(-1, 1)

relu = ReLU()
sigmoid = Sigmoid()

# Forward-Pass (Cache befüllen)
relu.forward(x)
sigmoid.forward(x)

# Backward-Pass mit dout=1 (reiner Gradient)
dout = np.ones_like(x)
grad_relu = relu.backward(dout)
grad_sigmoid = sigmoid.backward(dout)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, grad_relu, 'b-', linewidth=2)
axes[0].set_title("ReLU Gradient: f'(x) = 1(x>0)")
axes[0].set_xlabel('x')
axes[0].set_ylabel("f'(x)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, grad_sigmoid, 'g-', linewidth=2)
axes[1].set_title("Sigmoid Gradient: f'(x) = f(x)·(1-f(x))")
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Dense Layer — das Herzstück

Ein **Dense Layer** (vollständig verbundener Layer) berechnet:

$$\mathbf{y} = \mathbf{x} \cdot \mathbf{W} + \mathbf{b}$$

- $\mathbf{x}$: Input (Batch × input_dim)
- $\mathbf{W}$: Gewichtsmatrix (input_dim × output_dim)
- $\mathbf{b}$: Bias-Vektor (1 × output_dim)

**Initialisierung:** He-Initialisierung — $W \sim \mathcal{N}(0, \sqrt{2/\text{input\_dim}})$ — optimal für ReLU.

**Backward-Pass (Backpropagation):**
$$\frac{\partial L}{\partial \mathbf{x}} = \frac{\partial L}{\partial \mathbf{y}} \cdot \mathbf{W}^T$$
$$\frac{\partial L}{\partial \mathbf{W}} = \mathbf{x}^T \cdot \frac{\partial L}{\partial \mathbf{y}}$$
$$\frac{\partial L}{\partial \mathbf{b}} = \sum \frac{\partial L}{\partial \mathbf{y}}$$

In [ ]:
# ── Dense Layer live: Forward + Backward ──
np.random.seed(42)

# Erstelle einen Layer: 4 Input-Features → 3 Output-Neuronen
layer = Dense(input_dim=4, output_dim=3)

print("📐 Layer-Architektur:")
print(f"   W shape: {layer.W.shape}  (input_dim × output_dim)")
print(f"   b shape: {layer.b.shape}  (1 × output_dim)")
print(f"   He-Init std: {np.std(layer.W):.4f}  (theoretisch: {np.sqrt(2/4):.4f})")

# Forward-Pass
x = np.array([[1.0, 2.0, 3.0, 4.0],
              [0.5, 1.5, 2.5, 3.5]])
out = layer.forward(x)

print(f"\n📤 Forward-Pass:")
print(f"   Input shape:  {x.shape}")
print(f"   Output shape: {out.shape}")
print(f"   Output:\n{out}")

# Backward-Pass
dout = np.ones_like(out) * 0.1  # Simulierter Gradient von oben
dx = layer.backward(dout)

print(f"\n📥 Backward-Pass:")
print(f"   dout shape: {dout.shape}")
print(f"   dx shape:   {dx.shape}  (zurück zum Input)")
print(f"   dW shape:   {layer.dW.shape}  (Gewichts-Gradient)")
print(f"   db shape:   {layer.db.shape}  (Bias-Gradient)")
print(f"\n   dW:\n{layer.dW}")
print(f"   db:\n{layer.db}")

---
## 3. Cross-Entropy Loss

Für Klassifikation mit $C$ Klassen:

$$L = -\frac{1}{N} \sum_{i=1}^{N} \log\left(\frac{e^{z_{i,y_i}}}{\sum_{j=1}^{C} e^{z_{i,j}}}\right)$$

Wir kombinieren **Softmax + Cross-Entropy** in einem Schritt — das ist numerisch stabiler als beides getrennt zu berechnen.

Der Gradient der kombinierten Funktion ist erstaunlich einfach:
$$\frac{\partial L}{\partial \mathbf{z}} = \frac{1}{N}(\text{softmax}(\mathbf{z}) - \text{one\_hot}(\mathbf{y}))$$

In [ ]:
# ── Cross-Entropy Loss demonstrieren ──
loss_fn = CrossEntropyLoss()

# Perfekte Vorhersage: Loss ≈ 0
logits_perfect = np.array([[100.0, 0.0, 0.0],
                           [0.0, 100.0, 0.0],
                           [0.0, 0.0, 100.0]])
y_perfect = np.array([0, 1, 2])
loss_perfect = loss_fn.forward(logits_perfect, y_perfect)

# Schlechte Vorhersage: Loss hoch
logits_bad = np.array([[0.0, 0.0, 100.0],  # Sollte Klasse 0 sein!
                       [100.0, 0.0, 0.0],  # Sollte Klasse 1 sein!
                       [0.0, 100.0, 0.0]]) # Sollte Klasse 2 sein!
y_bad = np.array([0, 1, 2])
loss_bad = loss_fn.forward(logits_bad, y_bad)

print("📉 Cross-Entropy Loss:")
print(f"   Perfekte Vorhersage: {loss_perfect:.6f}  (≈ 0)")
print(f"   Schlechte Vorhersage: {loss_bad:.4f}  (deutlich höher)")

# Gradient visualisieren
loss_fn2 = CrossEntropyLoss()
logits = np.random.randn(5, 4)
y = np.array([0, 1, 2, 3, 1])
loss_fn2.forward(logits, y)
grad = loss_fn2.backward()

print(f"\n📊 Gradient Shape: {grad.shape}")
print(f"   Gradient summiert über Klassen (pro Sample ≈ 0): {np.sum(grad, axis=1)}")

---
## 4. SGD Optimizer mit Momentum

**Stochastic Gradient Descent** aktualisiert die Gewichte nach jedem Batch:

$$\mathbf{v}_{t+1} = \mu \cdot \mathbf{v}_t - \eta \cdot \nabla L$$
$$\mathbf{W}_{t+1} = \mathbf{W}_t + \mathbf{v}_{t+1}$$

- $\eta$: Lernrate (learning rate)
- $\mu$: Momentum (0.9 = 90% des vorherigen Updates wird beibehalten)

**Momentum** hilft:
- Schnellere Konvergenz
- Überwinden von lokalen Minima
- Stabileres Training

In [ ]:
# ── SGD mit vs. ohne Momentum ──
np.random.seed(42)

# Einfaches 2D-Problem: Gewichte von 1.0 auf 0.0 bewegen
layer_no_mom = Dense(2, 1)
layer_no_mom.W = np.ones((2, 1))
layer_no_mom.dW = np.ones((2, 1)) * 0.1

layer_mom = Dense(2, 1)
layer_mom.W = np.ones((2, 1))
layer_mom.dW = np.ones((2, 1)) * 0.1

opt_no_mom = SGD(lr=0.1, momentum=0.0)
opt_mom = SGD(lr=0.1, momentum=0.9)

w_no_mom = [layer_no_mom.W[0, 0]]
w_mom = [layer_mom.W[0, 0]]

for _ in range(20):
    opt_no_mom.step([layer_no_mom])
    opt_mom.step([layer_mom])
    w_no_mom.append(layer_no_mom.W[0, 0])
    w_mom.append(layer_mom.W[0, 0])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(w_no_mom, 'b-o', label='Ohne Momentum (μ=0)', markersize=4)
ax.plot(w_mom, 'r-o', label='Mit Momentum (μ=0.9)', markersize=4)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Update-Schritt')
ax.set_ylabel('Gewicht W[0,0]')
ax.set_title('SGD: Momentum beschleunigt die Konvergenz')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Das komplette NeuralNetwork

Jetzt fügen wir alles zusammen. Die `NeuralNetwork`-Klasse orchestriert:

1. **Forward-Pass:** Daten fließen durch alle Layer
2. **Loss-Berechnung:** Cross-Entropy
3. **Backward-Pass:** Gradienten fließen rückwärts
4. **Parameter-Update:** SGD mit Momentum

```
Input → Dense → ReLU → Dense → ReLU → Dense → Loss
  ↑                                               │
  └──────────── Backpropagation ◄──────────────────┘
```

In [ ]:
# ── Komplettes Netzwerk bauen ──
np.random.seed(42)

# Architektur: 784 → 128 → 64 → 10
net = NeuralNetwork([
    Dense(784, 128), ReLU(),
    Dense(128, 64),  ReLU(),
    Dense(64, 10),
])

# Parameter zählen
total_params = sum(
    layer.W.size + layer.b.size
    for layer in net.layers
    if isinstance(layer, Dense)
)

print("🏗️ Netzwerk-Architektur:")
print("   Input (784) → Dense(128) → ReLU → Dense(64) → ReLU → Dense(10) → Output")
print(f"\n📊 Parameter-Übersicht:")
for i, layer in enumerate(net.layers):
    if isinstance(layer, Dense):
        params = layer.W.size + layer.b.size
        print(f"   Layer {i} (Dense): W{list(layer.W.shape)} + b{list(layer.b.shape)} = {params:,} Parameter")
    else:
        print(f"   Layer {i} ({layer.__class__.__name__}): 0 Parameter (Aktivierung)")
print(f"\n   📌 Gesamt: {total_params:,} trainierbare Parameter")

# Test-Forward-Pass mit Zufallsdaten
x_test = np.random.randn(4, 784).astype(np.float32)
logits = net.forward(x_test)
probs = net.predict_proba(x_test)
preds = net.predict(x_test)

print(f"\n✅ Forward-Pass Test:")
print(f"   Input:  {x_test.shape}")
print(f"   Logits: {logits.shape}")
print(f"   Probs:  {probs.shape}  (Summe pro Zeile = 1: {np.allclose(probs.sum(axis=1), 1.0)})")
print(f"   Preds:  {preds}")

---
## 6. Training auf MNIST

Jetzt trainieren wir unser selbstgebautes Netzwerk auf dem echten MNIST-Datensatz (handgeschriebene Ziffern 0–9).

**Setup:**
- 60.000 Trainingsbilder, 10.000 Testbilder
- 28×28 Pixel = 784 Input-Features
- 10 Ausgabeklassen (Ziffern 0–9)
- Batch-Größe: 64, Epochen: 10

In [ ]:
# ── MNIST-Daten laden ──
import gzip
from urllib import request

def load_mnist():
    """Lädt MNIST aus lokalem Cache oder lädt es herunter."""
    cache_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".", ".mnist_cache")
    os.makedirs(cache_dir, exist_ok=True)

    files = {
        "train_images": "train-images-idx3-ubyte.gz",
        "train_labels": "train-labels-idx1-ubyte.gz",
        "test_images": "t10k-images-idx3-ubyte.gz",
        "test_labels": "t10k-labels-idx1-ubyte.gz",
    }
    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    for fname in files.values():
        path = os.path.join(cache_dir, fname)
        if not os.path.exists(path):
            print(f"  ⬇️ Lade {fname} herunter...")
            request.urlretrieve(base_url + fname, path)

    def load_images(path):
        with gzip.open(path, "rb") as f:
            data = np.frombuffer(f.read(), np.uint8, offset=16)
        return data.reshape(-1, 784).astype(np.float32) / 255.0

    def load_labels(path):
        with gzip.open(path, "rb") as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)

    X_train = load_images(os.path.join(cache_dir, files["train_images"]))
    y_train = load_labels(os.path.join(cache_dir, files["train_labels"]))
    X_test = load_images(os.path.join(cache_dir, files["test_images"]))
    y_test = load_labels(os.path.join(cache_dir, files["test_labels"]))

    return X_train, y_train, X_test, y_test

print("📦 Lade MNIST-Daten...")
X_train, y_train, X_test, y_test = load_mnist()
print(f"   Train: {X_train.shape[0]:,} Bilder, {X_train.shape[1]} Features")
print(f"   Test:  {X_test.shape[0]:,} Bilder")
print(f"   Klassen: {np.unique(y_train)}")
print(f"   Pixel-Wertebereich: [{X_train.min():.2f}, {X_train.max():.2f}]")

In [ ]:
# ── Einige MNIST-Bilder anzeigen ──
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(X_train))
    ax.imshow(X_train[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST Trainingsdaten — Zufällige Beispiele', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Training ──
np.random.seed(42)

# Netzwerk bauen
net = NeuralNetwork([
    Dense(784, 128), ReLU(),
    Dense(128, 64),  ReLU(),
    Dense(64, 10),
])

optimizer = SGD(lr=0.1, momentum=0.9)
batch_size = 64
epochs = 10

print("🏋️ Starte Training...")
print(f"   Architektur: 784 → 128 → 64 → 10")
print(f"   Batch-Größe: {batch_size}")
print(f"   Epochen: {epochs}")
print(f"   Optimizer: SGD (lr=0.1, momentum=0.9)")
print()

history = {"train_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(epochs):
    # Shuffle
    idx = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[idx]
    y_train_shuffled = y_train[idx]

    total_loss = 0
    total_acc = 0
    n_batches = 0

    for i in range(0, len(X_train_shuffled), batch_size):
        x_batch = X_train_shuffled[i : i + batch_size]
        y_batch = y_train_shuffled[i : i + batch_size]

        loss, acc = net.train_step(x_batch, y_batch, optimizer)
        total_loss += loss
        total_acc += acc
        n_batches += 1

    avg_loss = total_loss / n_batches
    avg_acc = total_acc / n_batches

    # Test-Accuracy
    test_preds = net.predict(X_test)
    test_acc = np.mean(test_preds == y_test)

    history["train_loss"].append(avg_loss)
    history["train_acc"].append(avg_acc)
    history["test_acc"].append(test_acc)

    print(f"   Epoche {epoch+1:2d}: Loss={avg_loss:.4f}  "
          f"Train-Acc={avg_acc:.3f}  Test-Acc={test_acc:.3f}")

print("\n✅ Training abgeschlossen!")

In [ ]:
# ── Trainingsverlauf visualisieren ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(range(1, epochs+1), history["train_loss"], 'b-o', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoche')
axes[0].set_ylabel('Loss')
axes[0].set_title('Trainings-Loss über die Epochen')
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, epochs+1), history["train_acc"], 'g-o', linewidth=2, markersize=6, label='Training')
axes[1].plot(range(1, epochs+1), history["test_acc"], 'r-s', linewidth=2, markersize=6, label='Test')
axes[1].set_xlabel('Epoche')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy über die Epochen')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 Finale Test-Accuracy: {history['test_acc'][-1]:.4f} ({history['test_acc'][-1]*100:.2f}%)")

---
## 7. Fehleranalyse

Welche Ziffern werden am häufigsten verwechselt? Wir schauen uns die falsch klassifizierten Bilder an.

In [ ]:
# ── Fehleranalyse ──
test_preds = net.predict(X_test)
errors = np.where(test_preds != y_test)[0]

print(f"🔍 Fehleranalyse:")
print(f"   Fehlklassifikationen: {len(errors)} / {len(y_test)} ({len(errors)/len(y_test)*100:.2f}%)")

# Confusion-Matrix (einfach, ohne seaborn)
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(y_test, test_preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues', aspect='auto')
for i in range(10):
    for j in range(10):
        text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=text_color, fontsize=9)
ax.set_xlabel('Vorhergesagt', fontsize=12)
ax.set_ylabel('Wahrheit', fontsize=12)
ax.set_title('Confusion Matrix — Unser selbstgebautes NN', fontsize=14, fontweight='bold')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Häufigste Verwechslungen
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
top_errors = np.unravel_index(np.argsort(off_diag.ravel())[-5:], cm.shape)
print("\n📌 Häufigste Verwechslungen:")
for t, p in zip(top_errors[0][::-1], top_errors[1][::-1]):
    print(f"   Wahrheit {t} → Vorhergesagt {p}: {cm[t, p]}×")

In [ ]:
# ── Falsch klassifizierte Bilder anzeigen ──
n_show = 10
error_idx = np.random.choice(errors, min(n_show, len(errors)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    if i < len(error_idx):
        idx = error_idx[i]
        ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
        ax.set_title(f'Wahr: {y_test[idx]} → Vorh.: {test_preds[idx]}',
                    color='red', fontsize=11)
    ax.axis('off')
plt.suptitle('Falsch klassifizierte Bilder', fontsize=14, fontweight='bold', color='red')
plt.tight_layout()
plt.show()

---
## 8. Zusammenfassung

Wir haben ein vollständiges neuronales Netzwerk **von Grund auf** gebaut — nur mit NumPy:

| Komponente | Implementierung |
|---|---|
| **Dense Layer** | `y = x @ W + b` mit He-Initialisierung |
| **ReLU** | `max(0, x)` — Standard-Aktivierung |
| **Sigmoid** | `1/(1+e^(-x))` — Alternative Aktivierung |
| **Softmax** | Wahrscheinlichkeitsverteilung für Klassifikation |
| **Cross-Entropy Loss** | Kombiniert Softmax+Loss für numerische Stabilität |
| **SGD + Momentum** | Beschleunigte Gradientenabstiege |
| **Backpropagation** | Kettenregel rückwärts durch alle Layer |

**Ergebnis auf MNIST:** Unser selbstgebautes Netz erreicht ~95% Test-Accuracy — ganz ohne Deep-Learning-Framework!

### Nächste Schritte:
- Experimentiere mit der Architektur (mehr/weniger Layer, andere Aktivierungen)
- Probiere andere Optimizer (Adam, RMSprop)
- Füge Dropout oder Batch-Normalization hinzu
- Trainiere auf anderen Datensätzen (Fashion-MNIST, CIFAR-10)